# Lab 7: Classification with PyTorch


# Neural networks

In this notebook we will learn how to train a simple Multilayer Perceptron and a simple Convolutional Neural Network (CNN) for image classification using PyTorch.

[Click here to check guide to install PyTorch locally.](https://pytorch.org/get-started/locally/)

You can find additional information [here](https://pytorch.org/tutorials/beginner/basics/intro.html).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn.functional as F
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as transforms
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from PIL import Image

## Load dataset

The torchvision package contains a few datasets. We will use the MNIST dataset of handwritten digits.

The dataset comes separated into training and test sets. We will further separate the test set into two smaller sets: validation and test.


In [ ]:
# Define transformations that are applied to the image
data_aug = transforms.Compose([transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)]) # ToTensor() transforms an image into a tensor, normalizing it into values between 0 and 1

# Load training data from MNIST into directory defined in "root"
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=data_aug,
)

# Load test data from MNIST into directory defined in "root"
validation_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=data_aug,
)

# Separate test images into validation (80%) and testing (20%)
indices = list(range(len(validation_data)))
np.random.shuffle(indices)

test_size = 0.2 * len(indices)
split = int(np.floor(test_size))
val_idx, test_idx = indices[split:], indices[:split]

val_sampler = SubsetRandomSampler(val_idx)
test_sampler = SubsetRandomSampler(test_idx)

print(f'Training size: {len(training_data)}\nValidation size: {len(val_sampler)} \nTest size: {len(test_sampler)}')

Define a data loader that automatically fetches batches of images and their labels

In [ ]:
batch_size = 64 # number of images loaded at each time
num_workers = 2 # how many processes are used to load the data

# Define data loaders for the train, test and validation data
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
validation_dataloader = DataLoader(validation_data, sampler=val_sampler, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)
test_dataloader = DataLoader(validation_data, sampler=test_sampler, batch_size=1, shuffle=False, num_workers=num_workers, drop_last=False)

## How to use datasets that are not available on torchvision?

Download dataset and upload them to drive or to the notebook.

You can download the dataset using the following link: [Download MNIST here!](https://git-disl.github.io/GTDLBench/datasets/mnist_datasets/) or as shown in the examples bellow.

For MNIST we will use the python.mnist package to read its files.

In [ ]:
!pip install python-mnist

**Option 1**: Upload dataset to Google Drive

First, we need to mount the drive on the colab notebook by running the following code and allowing access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Download the dataset from the link, unzip the content and place it in a folder in your Google Drive

In [ ]:
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id=11ZiNnV3YtpZ7d9afHZg0rtDRrmhha-1E" -O mnist.zip
!mkdir -p /content/drive/MyDrive/data_mnist
!unzip mnist.zip -d /content/drive/MyDrive/data_mnist/
!rm mnist.zip

Change the file names, since MNIST dataloader expects the files to be named like *train-images-idx3-ubyte* and not *train-images.idx3-ubyte*

In [ ]:
!mv /content/drive/MyDrive/data_mnist/train-images.idx3-ubyte /content/drive/MyDrive/data_mnist/train-images-idx3-ubyte
!mv /content/drive/MyDrive/data_mnist/train-labels.idx1-ubyte /content/drive/MyDrive/data_mnist/train-labels-idx1-ubyte
!mv /content/drive/MyDrive/data_mnist/t10k-images.idx3-ubyte /content/drive/MyDrive/data_mnist/t10k-images-idx3-ubyte
!mv /content/drive/MyDrive/data_mnist/t10k-labels.idx1-ubyte /content/drive/MyDrive/data_mnist/t10k-labels-idx1-ubyte

Then, we need to load the data from the correct path.

In [ ]:
from mnist import MNIST

# Load data using MNIST package - change the path to the folder where you saved the dataset on drive
mndata = MNIST("drive/MyDrive/data_mnist")

# Load images and labels from path
train_images, train_labels = mndata.load_training()
test_images, test_labels = mndata.load_testing()


**Option 2**: Upload dataset directly to Colab

Download the dataset from the link, unzip the content and place it in a Colab temporary folder

In [ ]:
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id=11ZiNnV3YtpZ7d9afHZg0rtDRrmhha-1E" -O mnist.zip
!mkdir -p data_mnist
!unzip mnist.zip -d data_mnist
!rm mnist.zip

Change the file names, since MNIST dataloader expects the files to be named like *train-images-idx3-ubyte* and not *train-images.idx3-ubyte*

In [ ]:
!mv data_mnist/train-images.idx3-ubyte data_mnist/train-images-idx3-ubyte
!mv data_mnist/train-labels.idx1-ubyte data_mnist/train-labels-idx1-ubyte
!mv data_mnist/t10k-images.idx3-ubyte data_mnist/t10k-images-idx3-ubyte
!mv data_mnist/t10k-labels.idx1-ubyte data_mnist/t10k-labels-idx1-ubyte

In [ ]:
from mnist import MNIST

# Load data using MNIST package - change the path to the folder where you saved the dataset on drive
mndata = MNIST('data_mnist')

# Load images and labels from path
train_images, train_labels = mndata.load_training()
test_images, test_labels = mndata.load_testing()

### Define a Custom Dataset class

Now that we have the images and labels, we can define a custom dataset class that can be used to retrieve and preprocess the data.

[Click here for more information on Datasets and Dataloaders.](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

In [ ]:
class MNISTCustomDataset(Dataset):
  def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
      image = self.images[idx]
      image = np.array(image, dtype=np.uint8).reshape((28, 28))

      # Transform array into grayscale image
      image = Image.fromarray(image, mode='L')

      # Apply transformations to the image
      if self.transform:
        image = self.transform(image)

      label = int(self.labels[idx])
      return (image, label)

# Define transformations
data_aug = transforms.Compose([transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)])

# Define custom dataset for the training and validation data
training_data = MNISTCustomDataset(train_images, train_labels, transform=data_aug)
validation_data = MNISTCustomDataset(test_images, test_labels, transform=data_aug)

# Show one image
plt.imshow(training_data[0][0][0,:,:], cmap='gray')

Divide validation set into validation (80%) and test (20%) subsets.

In [ ]:
indices = list(range(len(validation_data)))
np.random.shuffle(indices)

test_size = 0.2 * len(indices)
split = int(np.floor(test_size))
val_idx, test_idx = indices[split:], indices[:split]

val_sampler = SubsetRandomSampler(val_idx)
test_sampler = SubsetRandomSampler(test_idx)

print(f'Training size: {len(training_data)}\nValidation size: {len(val_sampler)} \nTest size: {len(test_sampler)}')

Define data loaders to automatically obtain batches of images to train the model

In [ ]:
batch_size = 64 # how many images are processed at a time
num_workers = 2 # how many processes are used to load the data

# Define data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
validation_dataloader = DataLoader(validation_data, sampler=val_sampler, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)
test_dataloader = DataLoader(validation_data, sampler=test_sampler, batch_size=1, shuffle=False, num_workers=num_workers, drop_last=False)

## Visualize the Data

Directly using the dataset

In [ ]:
# Get the first sample of the training data (contains an image and its label)

sample = training_data[0]

# Get image and print its dimensions
img = sample[0]
print(img.shape)

# Get label and print it
label = sample[1]
print(label)

Iterating over the data loader

In [ ]:
for batch in train_dataloader:
  # Get images of the batch and print their dimensions
  imgs = batch[0]
  print(imgs.shape)

  # Get labels of each image in the batch and print them
  labels = batch[1]
  print(labels)

  # Show first image of the batch
  plt.imshow(imgs[0][0,:,:], cmap='gray')
  plt.axis('off')
  plt.show()

  break

# MLP model

## Defining the MLP model

Create an MLP with the following structure:

1. Dense/linear layer that takes the images as a flattened input vector and generates an output of 512 of dimension.
2. ReLU activation layer
3. Dense/linear layer with 512 input and output
3. ReLU activation layer
4. Dense/linear layer with 10 output channels (10 classes of MNIST)

You can use PyTorch's layers: https://pytorch.org/docs/stable/nn.html (Conv2d, ReLU, Linear, MaxPool2d, Dropout, Flatten)

In [ ]:
# Get cpu or gpu device for training.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        # TODO: Define model layers here

    def forward(self, x):
      # TODO: Apply layers to input x and return result

model = NeuralNetwork().to(device) # put model in device (GPU or CPU)
print(model)

Interpret the implemented architecture and try to answer the following questions:

a) What is the shape (width, and # of channels) of the output tensor after the first layer?

b) And after the first 3 layers (dense+dense+dense)?

c) How many parameters (weights) does the model have? Contrary to Keras, PyTorch does not have an official method for counting the number of parameters of a model, but you can use [torchsummary](https://pypi.org/project/torch-summary/)

In [ ]:
#TODO

In [ ]:
#!pip install torch-summary

#TODO

## Train the model

In [ ]:
# Define loss function
loss_fn = nn.CrossEntropyLoss() # already includes the Softmax activation

# Define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

Define one iteration

In [ ]:
def epoch_iter(dataloader, model, loss_fn, optimizer=None, is_train=True):
    if is_train:
      assert optimizer is not None, "When training, please provide an optimizer."

    # Get number of batches
    num_batches = len(dataloader)

    # Set model to train mode or evaluation mode
    if is_train:
      model.train()
    else:
      model.eval()

    # Define variables to save predictions and labels during the epoch
    total_loss = 0.0
    preds = []
    labels = []

    # Enable/disable gradients based on whether the model is in train or evaluation mode
    with torch.set_grad_enabled(is_train):

      # Analyse all batches
      for batch, (X, y) in enumerate(tqdm(dataloader)):

          # Put data in same device as model (GPU or CPU)
          X, y = X.to(device), y.to(device)

          # Forward pass to obtain prediction of the model
          pred = model(X)

          # Compute loss between prediction and ground-truth
          loss = loss_fn(pred, y)

          # Backward pass
          if is_train:
            # Reset gradients in optimizer
            optimizer.zero_grad()
            # Calculate gradients by backpropagating loss
            loss.backward()
            # Update model weights based on the calculated gradients
            optimizer.step()

          # Apply softmax activation to obtain final prediction
          probs = F.softmax(pred, dim=1)
          final_pred = torch.argmax(probs, dim=1)

          # Save training metrics
          total_loss += loss.item() # IMPORTANT: call .item() to obtain the value of the loss WITHOUT the computational graph attached

          # Add predictions
          preds.extend(final_pred.cpu().numpy())
          labels.extend(y.cpu().numpy())

    return total_loss / num_batches, accuracy_score(labels, preds)

Define training cycle

In [ ]:
num_epochs = 10
train_history = {'loss': [], 'accuracy': []}
val_history = {'loss': [], 'accuracy': []}
best_val_loss = np.inf

# Training cycle
print("Start training...")
for t in range(num_epochs):
    print(f"\nEpoch {t+1}")

    # Train model for one iteration on training data
    train_loss, train_acc = epoch_iter(train_dataloader, model, loss_fn, optimizer)
    print(f"Train loss: {train_loss:.3f} \t Train acc: {train_acc:.3f}")

    # Evaluate model on validation data
    val_loss, val_acc = epoch_iter(validation_dataloader, model, loss_fn, is_train=False)
    print(f"Val loss: {val_loss:.3f} \t Val acc: {val_acc:.3f}")

    # Save model when validation loss improves
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      save_dict = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': t}
      torch.save(save_dict, 'best_model.pth')

    # Save latest model
    save_dict = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': t}
    torch.save(save_dict, 'latest_model.pth')

    # Save training history for plotting purposes
    train_history["loss"].append(train_loss)
    train_history["accuracy"].append(train_acc)

    val_history["loss"].append(val_loss)
    val_history["accuracy"].append(val_acc)

print("Finished")

## Analyse training evolution

Plot loss and accuracy throughout training on train and validation data

In [ ]:
#TODO

## Test the model

Evaluate the model in the test set

In [ ]:
# TODO

In [ ]:
def showErrors(model, dataloader, num_examples=20):
    plt.figure(figsize=(15, 15))

    for ind, (X, y) in enumerate(dataloader):
      if ind >= 20: break
      X, y = X.to(device), y.to(device)
      pred = model(X)
      probs = F.softmax(pred, dim=1)
      final_pred = torch.argmax(probs, dim=1)

      plt.subplot(10, 10, ind + 1)
      plt.axis("off")
      plt.text(0, -1, y[0].item(), fontsize=14, color='green') # correct
      plt.text(8, -1, final_pred[0].item(), fontsize=14, color='red')  # predicted
      plt.imshow(X[0][0,:,:].cpu(), cmap='gray')
    plt.show()

showErrors(model, test_dataloader)

# CNN model

## Defining the CNN model

Create a CNN with the following structure:

1. convolutional layer with 32 output channels and 3x3 kernel
2. ReLU activation layer
3. convolutional layer with 32 input/output channels and 3x3 kernel
4. ReLU activation layer
5. max pooling layer with a kernel size of 2
6. dropout with 0.25 probability
7. flattening layer (to convert the 3D tensors into 1D tensors to be fed to the dense layers)
8. dense/linear layer with 128 output channels
9. ReLU activation layer
10. dropout layer with 0.5 probability
11. dense/linear layer with 10 output channels (10 classes of MNIST)

You can use PyTorch's layers: https://pytorch.org/docs/stable/nn.html (Conv2d, ReLU, Linear, MaxPool2d, Dropout, Flatten)




In [ ]:
class ConvolutionalNeuralNetwork(nn.Module):
    def __init__(self):
        super(ConvolutionalNeuralNetwork, self).__init__()
        self.pool_size = 2
        self.nb_filters = 32
        self.kernel_size = 3

        # TODO: Define model layers here

    def forward(self, x):
        # TODO: Apply layers to input x and return result

model = ConvolutionalNeuralNetwork().to(device) # put model in device (GPU or CPU)
print(model)

Interpret the implemented architecture and try to answer the following questions:

a) What is the shape (width, height and # of channels) of the output tensor after the first convolution layer?

b) And after the first 3 layers (convolution+convolution+pooling)?

c) How many parameters (weights) does the model have? Contrary to Keras, PyTorch does not have an official method for counting the number of parameters of a model, but you can use [torchsummary](https://github.com/sksq96/torchsummary).

In [ ]:
# !pip install torchsummary
#TODO

## Train the model

Define loss function and optimizer

In [ ]:
# Define loss function
#TODO

# Define optimizer
#TODO

Define one epoch of the model

In [ ]:
def epoch_iter(dataloader, model, loss_fn, optimizer=None, is_train=True):
    if is_train:
      assert optimizer is not None, "When training, please provide an optimizer."

    num_batches = len(dataloader)

    if is_train:
      model.train() # put model in train mode
    else:
      model.eval()

    total_loss = 0.0
    preds = []
    labels = []

    with torch.set_grad_enabled(is_train):
      for batch, (X, y) in enumerate(tqdm(dataloader)):
          X, y = X.to(device), y.to(device)

          # Obtain prediction
          # pred = TODO

          # Obtain loss value
          # loss = TODO

          if is_train:
            # Backpropagation
            # TODO

          # Save training metrics
          total_loss += loss.item() # IMPORTANT: call .item() to obtain the value of the loss WITHOUT the computational graph attached

          # Calculate the final prediction
          # final_pred = TODO
          preds.extend(final_pred.cpu().numpy())
          labels.extend(y.cpu().numpy())

    return total_loss / num_batches, accuracy_score(labels, preds)

Train the model

In [ ]:
num_epochs = 10
train_history = {'loss': [], 'accuracy': []}
val_history = {'loss': [], 'accuracy': []}
best_val_loss = np.inf
print("Start training...")
for t in range(num_epochs):
    print(f"\nEpoch {t+1}")
    train_loss, train_acc = epoch_iter(train_dataloader, model, loss_fn, optimizer)
    print(f"Train loss: {train_loss:.3f} \t Train acc: {train_acc:.3f}")
    val_loss, val_acc = epoch_iter(validation_dataloader, model, loss_fn, is_train=False)
    print(f"Val loss: {val_loss:.3f} \t Val acc: {val_acc:.3f}")

    # save model when val loss improves
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      save_dict = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': t}
      torch.save(save_dict, 'best_model.pth')

    # save latest model
    save_dict = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': t}
    torch.save(save_dict, 'latest_model.pth')

    # save training history for plotting purposes
    train_history["loss"].append(train_loss)
    train_history["accuracy"].append(train_acc)

    val_history["loss"].append(val_loss)
    val_history["accuracy"].append(val_acc)

print("Finished")

## Analyse training evolution

Plot loss and accuracy throughout training on train and validation data

## Test the model

Load the model that performed the best on the validation data and evaluate it

In [1]:
# Load model
# TODO

# Test model
# TODO

Show examples of results

In [ ]:
showErrors(model, test_dataloader)

## Additional Challenges
Exercise 1

a) As the test accuracy should show, the MNIST dataset is not very challenging, change the code to use Fashion-MNIST and compare the results.

b) Do the same for the CIFAR10 (or CIFAR100) dataset. Note that, in this case, each image is a 32x32 color image; convert it to grayscale or concatenate the RGB channels in one single vector (e.g. using the reshape method).

c) The test accuracy for CIFAR is significantly worse. Try improving the results by using: 1) a deeper architecture, and 2) a different optmizer.

You can load the datasets from [here](https://pytorch.org/vision/stable/datasets.html).


Exercise 2

a) What is data augmentation and why is it useful? Explore some data augmentation techniques, by using some transforms from [torchvision](https://pytorch.org/vision/stable/transforms.html).

b) Since training a complex model can take a very long time to train, model checkpoints can be saved and loaded later to resume the training. Explore how this can be done: https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html

c) Train and test the previous model on the Fashion-MNIST and CIFAR-10 datasets. Some adaptations to the code are necessary for the latter dataset.
